In [ ]:
# Import all necessary packages
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import distinctipy
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
from plottable import ColumnDefinition, Table
import yaml
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/cytokines-regression"
df_feats = pd.read_excel(f"{path_data}/features.xlsx")
imms = df_feats.columns.to_list()
df = pd.read_excel(f"{path_data}/data.xlsx")

# Plot Figure 2b  
n_rows = 8 * 3
n_cols = 4
fig_height = 40
fig_width = 17

imm_colors = distinctipy.get_colors(n_colors=len(imms), exclude_colors=[mcolors.hex2color(mcolors.CSS4_COLORS['gray'])], rng=42)

sns.set_theme(style='ticks')
fig, axs = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), height_ratios=[0.2, 0.8, 0.2]*8, gridspec_kw={'wspace':0.35, 'hspace': 0.05}, sharey=False, sharex=False)

for imm_id, imm in enumerate(imms):
    imm_color = imm_colors[imm_id]
    imm_metrics = pd.read_excel(f"{path_root}/models/InflammatoryMarkers/{imm}/metrics.xlsx", index_col=0)
    with open(f"{path_root}/models/InflammatoryMarkers/{imm}/config.yml") as f:
        imm_config = yaml.safe_load(f)
    imm_df = pd.read_excel(f"{path_root}/models/InflammatoryMarkers/{imm}/df.xlsx", index_col=0)
    imm_df.rename(columns={f"{imm}_log": imm}, inplace=True)

    row_id, col_id = divmod(imm_id, n_cols)
    row_id_table = row_id * 3
    row_id_scatter = row_id * 3 + 1
    row_id_empty = row_id * 3 + 2

    q01 = df[imm].quantile(0.01)
    q99 = df[imm].quantile(0.99)

    df_metrics = pd.DataFrame(index=["Pearson's R"], columns=['Train', 'Validation', 'Test'])
    df_metrics.at["Pearson's R", 'Train'] = f"{imm_metrics.at['Train', 'pearson_corrcoef']:0.3f}"
    df_metrics.at["Pearson's R", 'Validation'] = f"{imm_metrics.at['Validation', 'pearson_corrcoef']:0.3f}"
    df_metrics.at["Pearson's R", 'Test'] = f"{imm_metrics.at['Test', 'pearson_corrcoef']:0.3f}"
    
    col_defs = [
        ColumnDefinition(
            name="index",
            title=imm_config['_model_name'].replace('Model', ''),
            textprops={"ha": "center", "weight": "bold"},
            width=2.5,
            group=fr"$\mathbf{{{imm}}}$",
        ),
        ColumnDefinition(
            name="Train",
            textprops={"ha": "left"},
            width=1.5,
            border="left",
            group=fr"$\mathbf{{{imm}}}$",
        ),
        ColumnDefinition(
            name="Validation",
            textprops={"ha": "left"},
            width=1.5,
            group=fr"$\mathbf{{{imm}}}$",
        ),
        ColumnDefinition(
            name="Test",
            textprops={"ha": "left"},
            width=1.5,
            group=fr"$\mathbf{{{imm}}}$",
        )
    ]

    table = Table(
        df_metrics,
        column_definitions=col_defs,
        row_dividers=True,
        footer_divider=False,
        ax=axs[row_id_table, col_id],
        textprops={"fontsize": 8},
        row_divider_kw={"linewidth": 1, "linestyle": (0, (1, 1))},
        col_label_divider_kw={"linewidth": 1, "linestyle": "-"},
        column_border_kw={"linewidth": 1, "linestyle": "-"},
    ).autoset_fontcolors(colnames=['Train', 'Validation', 'Test'])

    kdeplot = sns.kdeplot(
        data=imm_df.loc[imm_df['Group'] != 'Test', :],
        x=imm,
        y='Prediction',
        fill=True,
        cbar=False,
        color=imm_color,
        cut=0,
        legend=False,
        ax=axs[row_id_scatter, col_id]
    )
    scatter = sns.scatterplot(
        data=imm_df.loc[imm_df['Group'] == 'Test', :],
        x=imm,
        y="Prediction",
        linewidth=0.5,
        alpha=0.8,
        edgecolor="k",
        s=35,
        color=imm_color,
        ax=axs[row_id_scatter, col_id],
    )
    axs[row_id_scatter, col_id].axline((0, 0), slope=1, color="black", linestyle=":")
    axs[row_id_scatter, col_id].set_xlim(q01, q99)
    axs[row_id_scatter, col_id].set_ylim(q01, q99)
    axs[row_id_scatter, col_id].set_xlabel(imm, color=imm_color, path_effects=[pe.withStroke(linewidth=1.0, foreground="black")])
    
    axs[row_id_empty, col_id].axis('off')

fig.tight_layout()
fig.savefig(f"{path_plots}/figure2b.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_plots}/figure2b.pdf", bbox_inches='tight')
plt.close(fig)